# Utils

In [ ]:
#import pckgs
import cv2
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os
from natsort import natsorted
import re

### Delete Files with a specific string in directory 
Always run it with #os.remove commented first!

In [ ]:
main_path='/Volumes/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/'

for roots, dirs, files in natsorted(os.walk(main_path)):
    for single_dir in dirs:
        for single_file in natsorted(files):
#            print((os.path.join(roots,single_dir,single_file)))
            #print(files)
            if 'compressed' in single_file and 'ome.tif' not in single_file:
                print(os.path.join(roots, single_file))
                #os.remove(os.path.join(roots, single_file))
                #break

### ometiff2bigtiff
The function reads all the ometiff files in a directory and saves them in a big tiff file with the name of the directory + bigtiff.btf
It uses Sequential reading as Lukas showed me.

The code below is based on a notebook with the same name we wrote which is stored in /shared_projects/notebooks/lukas_notebooks

We didnt manage to save as .avi the big tiff file and decided to do it with fiji instead, which seems to work.

In [ ]:
#function to list all ome.tiff in a directory and make them one bif ome tif

#somehow it gives an error for the last ome tiff, but the movie is fine.
def ometiff2bigtiff(path):
    if path.endswith('/'):
        output_filename=path+re.split('/',path)[-2]+'bigtiff2.btf'
    else:
        output_filename=path+'/'+re.split('/',path)[-1]+'bigtiff2.btf'
    with tiff.TiffWriter(output_filename, bigtiff=True) as output_tif:
        for file in natsorted(os.listdir(path)):
            #print(os.path.join(path,file))
            if file.endswith('MMStack.ome.tif') and 'bg' not in file:
                print(os.path.join(path,file))
                with tiff.TiffFile(os.path.join(path,file)) as tif:
                    print('entered writing')
                    hyperstack = tif.asarray()
                    omexmlMetadataString = tif.ome_metadata
                    print('writing...')
                    output_tif.save(hyperstack, photometric='minisblack', description=omexmlMetadataString)

In [ ]:
main_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200605/'

for roots, dirs, files in natsorted(os.walk(main_path)):
    print(dirs)
    for single_dir in natsorted(dirs):
        if 'worm' in single_dir and 'bg' not in single_dir:
            print('the directory is:')
            print(os.path.join(roots,single_dir)+'\n')
            ometiff2bigtiff(os.path.join(roots,single_dir))

### Compress ometiff file


In [ ]:
#define compress_ometif function
def compress_ometif(file):
    #print('reading ome tif file: '+file+'\n... might take a while')
    retval, mats=cv2.imreadmulti(file)
    #print('reading complete')
    ometif=np.asarray(mats)
    new_img=np.full((ometif.shape[0],200,200), 0, dtype='uint8')
    #print('entering the loop')
    for k,img in enumerate(ometif):
        resized = cv2.resize(img, (200,200), interpolation = cv2.INTER_AREA)
        new_img[k]=resized
    return new_img
    #tiff.imsave(filename,new_img, bigsize=True) 

In [ ]:
#generate avi files for the ome.tif files in the directory
main_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200701/'

#for every root directory
for roots, dirs, files in os.walk(main_path):
    #for every file in roots
    for file in natsorted(files):
        if file.endswith('ome.tif') and 'bg' not in file:
            #print(os.path.join(roots,file))
            new_img=compress_ometif(os.path.join(roots,file))
            tiff.imsave(os.path.join(roots, re.split('.ome.tif',file)[0])+ '_compressed.tiff', new_img, bigsize=True)
print('end')
        

In [ ]:
#TEST merge all files in roots
print('!!!before running this I changed the Name of ...MMStack_compressed to ...MMStack_0_compressed in NameChanger\n\n\n')
main_path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200701/2020-07-01_18-36-25_control_worm6-channel-0-/'
for roots, dirs, files in natsorted(os.walk(main_path)):
    if 'worm' in roots and 'bg' not in roots:
        output_video=np.full((0,200,200), 0, dtype='uint8')
        #for every file in roots
        for file in natsorted(files):
            if file.endswith('.avi'):
                print(os.path.join(roots,file))
                mats=tiff.imread(os.path.join(roots,file))
                avi_file=np.asarray(mats)
                output_video=np.concatenate([output_video,avi_file],0)
                #print(output_video.shape)
        print(os.path.join(roots,re.split('/',roots)[-1])+'_ALL_compressed.tiff')
        tiff.imsave(os.path.join(roots,re.split('/',roots)[-1])+'_ALL_compressed.tiff', output_video, bigsize=True)
print('end')

In [ ]:
path='/groups/zimmer/Ulises/wbfm/chemotaxis_assay/2020_Only_behaviour/datasets/dataset_20200605/\
2020-06-05_16-46-41_worm1-channel-0-/'

output_video=np.full((0,200,200), 0, dtype='uint8')

for file in natsorted(os.listdir(path)):
    print(file)
    if file.endswith('ome.tif'):
        print(file)
        new_img=compress_ometif(path,file)
        output_video=np.concatenate([output_video,new_img],0)
tiff.imsave(path+'ourput.tiff',output_video, bigsize=True)
print('end')